# IoT Predictive Maintenance — Local Validation Reference

This notebook is the **pandas reference implementation** for the Gold-layer transformation logic (rolling averages, baseline comparison, health flagging) built in Azure Data Factory.

**Why this exists:** while debugging the ADF Data Flow, three real bugs were found and fixed — but confirming each fix required an independent, trusted source of truth to compare against. This notebook is that source of truth: the same logic, implemented in pandas, run locally and verified against known ground truth (the deliberately-injected machine degradation rates) before being used to validate the ADF pipeline's output.


## 1. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np

machines = pd.read_csv('machines.csv', parse_dates=['install_date'])
sensor = pd.read_csv('sensor_readings.csv', parse_dates=['timestamp'])
maintenance = pd.read_csv('maintenance_logs.csv', parse_dates=['maintenance_date'])

sensor = sensor.sort_values(['machine_id', 'timestamp']).reset_index(drop=True)
print(machines.shape, sensor.shape, maintenance.shape)

## 2. Rolling Average

For each machine, compute a rolling average over its last 4 readings (roughly the last day's worth of behavior, given 4 readings/day) - this represents how the machine is behaving right now.

In [ ]:
sensor['rolling_avg_temp'] = sensor.groupby('machine_id')['temperature_c'].transform(
    lambda x: x.rolling(4, min_periods=1).mean()
)
sensor['rolling_avg_vibration'] = sensor.groupby('machine_id')['vibration_mm_s'].transform(
    lambda x: x.rolling(4, min_periods=1).mean()
)
sensor['rolling_avg_pressure'] = sensor.groupby('machine_id')['pressure_psi'].transform(
    lambda x: x.rolling(4, min_periods=1).mean()
)

## 3. Baseline (first 15 days only)

**Design decision:** the baseline must represent each machine's known-healthy state, not its full history, which would include already-degraded readings and contaminate the reference point. Restricting to the first 15 days (the earliest data, before meaningful drift has accumulated) avoids this.

In [ ]:
cutoff = sensor['timestamp'].min() + pd.Timedelta(days=15)
early = sensor[sensor['timestamp'] <= cutoff]

baseline = early.groupby('machine_id').agg(
    baseline_avg_temp=('temperature_c', 'mean'),
    baseline_avg_vibration=('vibration_mm_s', 'mean'),
    baseline_avg_pressure=('pressure_psi', 'mean')
).reset_index()

print(f"Baseline table: {baseline.shape[0]} rows (should be exactly {sensor['machine_id'].nunique()}, one per machine)")
baseline

## 4. Join Baseline Back to Full Stream

In [ ]:
gold = sensor.merge(baseline, on='machine_id', how='left')
print(f"Gold rows: {len(gold)} (should exactly match source sensor row count: {len(sensor)})")

## 5. Deviation & Health Status

Compare recent behavior (rolling average) against the machine's own healthy baseline. Flag Watch if either temperature or vibration deviates meaningfully - an OR condition, since a machine can show early warning signs in just one sensor reading, not necessarily both simultaneously.

In [ ]:
gold['temp_deviation'] = gold['rolling_avg_temp'] - gold['baseline_avg_temp']
gold['vibration_deviation'] = gold['rolling_avg_vibration'] - gold['baseline_avg_vibration']

gold['health_status'] = np.where(
    (gold['temp_deviation'] > 3) | (gold['vibration_deviation'] > 0.5),
    'Watch', 'Normal'
)

gold['health_status'].value_counts(normalize=True) * 100

## 6. Ground-Truth Validation

The synthetic data generation deliberately assigned each machine a degradation_rate (0.0 for stable machines, up to 0.15 for the fastest-degrading). This is the actual ground truth - if the health-flagging logic works correctly, Watch rate should scale with degradation_rate.

In [ ]:
gold_with_degr = gold.merge(machines[['machine_id', 'degradation_rate']], on='machine_id', how='left')

watch_by_machine = gold_with_degr.groupby(['machine_id', 'degradation_rate'])['health_status'].apply(
    lambda x: (x == 'Watch').mean() * 100
).reset_index(name='watch_pct')

watch_by_machine.sort_values('degradation_rate', ascending=False)

**Result:** Watch percentage tracks degradation_rate closely - the three machines with degradation_rate = 0.15 show the highest Watch rates (9-14%), while stable machines (degradation_rate = 0.0) show under 1%. This confirms the logic correctly recovers the injected ground truth, and this output is what was used to validate the Azure Data Factory pipeline's Gold layer, row by row, until they matched to floating-point precision.

Full debugging narrative - the three bugs found and fixed in the ADF version of this same logic - is documented in the main README.